# QLoRA Spider benchmark: base versus 3-epoch adapter
100 identical validation examples; original schema-free training prompt; strict Pandas result equality.
Credentials are supplied by Kaggle, never embedded in notebook source.


In [ ]:
import os, sys, subprocess
os.environ["HF_HOME"] = "/tmp/hf-benchmark-cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
    "transformers==5.17.0", "peft==0.21.2", "datasets==4.8.5", "tokenizers==0.23.3",
    "accelerate>=1.0", "bitsandbytes>=0.45", "safetensors>=0.5", "sqlglot==30.30.0",
    "pandas>=2.2"], check=True)


In [ ]:
from pathlib import Path
import sys
script_dir = Path("/kaggle/working/benchmark_scripts")
script_dir.mkdir(exist_ok=True)
(script_dir / 'evaluate_spider.py').write_text('#!/usr/bin/env python3\n"""Generate SQL with a PEFT adapter and score it on Spider SQLite databases.\n\nThis reports single-database result equivalence, NOT official Spider test-suite\naccuracy. Dependencies for scoring: sqlglot. Generation additionally requires\ntorch, transformers, peft, accelerate, bitsandbytes, datasets and safetensors.\n"""\nimport argparse\nfrom collections import Counter\nfrom contextlib import closing\nimport importlib.metadata\nimport json\nfrom pathlib import Path\nimport random\nimport re\nimport sqlite3\nimport time\n\nimport sqlglot\nfrom sqlglot import exp\n\nSYSTEM_PROMPT = "You are an expert SQL assistant. Given a database schema and a question, write a valid SQL query that answers the question."\n\n\ndef extract_sql(text):\n    """Accept a single fenced SQL answer or a bare query; do not repair SQL."""\n    fences = re.findall(r"```(?:sql|sqlite)?\\s*\\n?(.*?)```", text, re.I | re.S)\n    if len(fences) > 1:\n        raise ValueError("Multiple code blocks in answer")\n    return (fences[0] if fences else text).strip()\n\n\ndef parse_query(sql):\n    statements = [s for s in sqlglot.parse(sql, read="sqlite") if s is not None]\n    if len(statements) != 1 or not isinstance(statements[0], exp.Query):\n        raise ValueError("Expected exactly one SELECT/compound query")\n    return statements[0]\n\n\ndef database_path(root, db_id):\n    if not re.fullmatch(r"[A-Za-z0-9_-]+", db_id):\n        raise ValueError(f"Invalid database ID: {db_id!r}")\n    path = (root / db_id / f"{db_id}.sqlite").resolve()\n    if not path.is_relative_to(root.resolve()) or not path.is_file():\n        raise FileNotFoundError(f"Missing Spider database: {path}")\n    return path\n\n\ndef execute_sql(path, sql, timeout=5.0, max_rows=100000):\n    """Read-only SQLite execution with an authorizer, VM timeout and row cap."""\n    started = time.monotonic()\n    allowed = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ,\n               sqlite3.SQLITE_FUNCTION, sqlite3.SQLITE_RECURSIVE}\n\n    def authorize(action, arg1, arg2, db, source):\n        if action == sqlite3.SQLITE_FUNCTION and (arg2 or "").lower() == "load_extension":\n            return sqlite3.SQLITE_DENY\n        return sqlite3.SQLITE_OK if action in allowed else sqlite3.SQLITE_DENY\n\n    with closing(sqlite3.connect(path.as_uri() + "?mode=ro", uri=True, timeout=timeout)) as conn:\n        conn.execute("PRAGMA query_only=ON")\n        conn.set_authorizer(authorize)\n        conn.set_progress_handler(lambda: int(time.monotonic() - started > timeout), 1000)\n        cursor = conn.execute(sql)\n        rows = cursor.fetchmany(max_rows + 1)\n        if len(rows) > max_rows:\n            raise ValueError(f"Result exceeds {max_rows} rows")\n        return rows, len(cursor.description or ())\n\n\ndef equivalent(gold, pred, ordered):\n    gold_rows, gold_width = gold\n    pred_rows, pred_width = pred\n    if gold_width != pred_width:\n        return False\n    return gold_rows == pred_rows if ordered else Counter(gold_rows) == Counter(pred_rows)\n\n\ndef score_one(path, gold_sql, answer, timeout=5.0, max_rows=100000):\n    record = {"prediction": "", "syntax_valid": False, "execution_valid": False,\n              "execution_match": False, "gold_valid": False}\n    try:\n        record["prediction"] = extract_sql(answer)\n        parse_query(record["prediction"])\n        record["syntax_valid"] = True\n    except (ValueError, sqlglot.errors.SqlglotError) as exc:\n        record["syntax_error"] = str(exc)\n    try:\n        gold_ast = parse_query(gold_sql)\n        gold = execute_sql(path, gold_sql, timeout, max_rows)\n        record["gold_valid"] = True\n    except (ValueError, sqlite3.Error, sqlglot.errors.SqlglotError) as exc:\n        record["gold_error"] = str(exc)\n        gold = None\n    if record["syntax_valid"]:\n        try:\n            pred = execute_sql(path, record["prediction"], timeout, max_rows)\n            record["execution_valid"] = True\n            if gold is not None:\n                record["execution_match"] = equivalent(gold, pred, gold_ast.args.get("order") is not None)\n        except (ValueError, sqlite3.Error) as exc:\n            record["execution_error"] = str(exc)\n    return record\n\n\ndef read_json_rows(path):\n    text = path.read_text()\n    return json.loads(text) if text.lstrip().startswith("[") else [json.loads(line) for line in text.splitlines() if line.strip()]\n\n\ndef schema_text(path):\n    with closing(sqlite3.connect(path.as_uri() + "?mode=ro", uri=True)) as conn:\n        return "\\n".join(row[0] + ";" for row in conn.execute(\n            "SELECT sql FROM sqlite_master WHERE type=\'table\' AND name NOT LIKE \'sqlite_%\' AND sql IS NOT NULL ORDER BY name"))\n\n\ndef load_generator(args):\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed\n    from peft import PeftConfig, PeftModel\n\n    if not torch.cuda.is_available():\n        raise RuntimeError("Generation requires a CUDA GPU. Use --predictions for CPU-only scoring.")\n    set_seed(args.seed)\n    config = PeftConfig.from_pretrained(args.adapter)\n    base = args.base_model or config.base_model_name_or_path\n    tokenizer = AutoTokenizer.from_pretrained(args.adapter)\n    model = AutoModelForCausalLM.from_pretrained(\n        base, device_map="auto", torch_dtype=torch.float16,\n        quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                                               bnb_4bit_use_double_quant=True,\n                                               bnb_4bit_compute_dtype=torch.float16))\n    model = PeftModel.from_pretrained(model, args.adapter).eval()\n    torch.cuda.reset_peak_memory_stats()\n\n    def generate(row, path):\n        prompt = f"Database: {row[\'db_id\']}\\n"\n        if args.schema == "ddl":\n            prompt += f"Schema:\\n{schema_text(path)}\\n"\n        prompt += f"Question: {row[\'question\']}"\n        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]\n        inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,\n                                               return_tensors="pt", return_dict=True)\n        inputs = {key: value.to(model.get_input_embeddings().weight.device) for key, value in inputs.items()}\n        prompt_tokens = inputs["input_ids"].shape[1]\n        if prompt_tokens + args.max_new_tokens > args.max_context:\n            raise ValueError(f"Prompt plus output budget exceeds {args.max_context} tokens; no silent truncation")\n        started = time.monotonic()\n        with torch.inference_mode():\n            output = model.generate(**inputs, max_new_tokens=args.max_new_tokens, do_sample=False,\n                                    use_cache=True, pad_token_id=tokenizer.eos_token_id)\n        tokens = output[0, prompt_tokens:]\n        return tokenizer.decode(tokens, skip_special_tokens=True), {\n            "generation_seconds": time.monotonic() - started,\n            "prompt_tokens": prompt_tokens, "generated_tokens": len(tokens),\n            "hit_token_limit": len(tokens) == args.max_new_tokens,\n        }\n\n    return generate, base\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    source = parser.add_mutually_exclusive_group(required=True)\n    source.add_argument("--adapter", help="Local final adapter directory or Hub adapter repository")\n    source.add_argument("--predictions", type=Path, help="JSON/JSONL rows with index, db_id, question, prediction")\n    parser.add_argument("--db-root", type=Path, required=True, help="Spider database/ directory")\n    parser.add_argument("--data", type=Path, help="Local Spider dev.json; defaults to HF validation split")\n    parser.add_argument("--dataset-revision", default="main", help="Use a commit SHA for reproducibility")\n    parser.add_argument("--base-model", help="Override base recorded in adapter config")\n    parser.add_argument("--schema", choices=["none", "ddl"], default="none")\n    parser.add_argument("--limit", type=int, help="Seeded random validation subset; omit for full validation")\n    parser.add_argument("--seed", type=int, default=3407)\n    parser.add_argument("--max-new-tokens", type=int, default=256)\n    parser.add_argument("--max-context", type=int, default=2048)\n    parser.add_argument("--timeout", type=float, default=5)\n    parser.add_argument("--max-rows", type=int, default=100000)\n    parser.add_argument("--output", type=Path, default=Path("spider_evaluation"))\n    args = parser.parse_args()\n    for name in ("timeout", "max_rows", "max_new_tokens", "max_context"):\n        if getattr(args, name) <= 0:\n            parser.error(f"--{name.replace(\'_\', \'-\')} must be positive")\n    if args.data:\n        rows = read_json_rows(args.data)\n    else:\n        from datasets import load_dataset\n        rows = list(load_dataset("xlangai/spider", split="validation", revision=args.dataset_revision))\n    indices = list(range(len(rows)))\n    if args.limit is not None:\n        if not 0 < args.limit <= len(rows):\n            parser.error("--limit must be between 1 and dataset size")\n        indices = sorted(random.Random(args.seed).sample(indices, args.limit))\n    if not indices:\n        parser.error("Dataset is empty")\n    args.output.mkdir(parents=True, exist_ok=True)\n    if any((args.output / name).exists() for name in ("predictions.jsonl", "summary.json", "gold.txt", "predicted.sql")):\n        parser.error("Output already contains evaluation results; choose a new --output directory")\n    # Validate every database before loading a costly model or reporting metrics.\n    paths = {rows[i]["db_id"]: database_path(args.db_root, rows[i]["db_id"]) for i in indices}\n    saved = {}\n    if args.predictions:\n        for item in read_json_rows(args.predictions):\n            index = item["index"]\n            if index in saved:\n                parser.error(f"Duplicate prediction index {index}")\n            saved[index] = item\n        for i in indices:\n            if i not in saved or any(saved[i].get(k) != rows[i][k] for k in ("db_id", "question")):\n                parser.error(f"Missing or misaligned prediction at validation index {i}")\n            if not isinstance(saved[i].get("prediction"), str):\n                parser.error(f"Prediction at index {i} must be a string")\n        generator, base = None, None\n    else:\n        generator, base = load_generator(args)\n    results = []\n    started = time.monotonic()\n    with (args.output / "predictions.jsonl").open("w") as out, (args.output / "gold.txt").open("w") as gold_out, (args.output / "predicted.sql").open("w") as sql_out:\n        for i in indices:\n            row, metadata = rows[i], {}\n            path = paths[row["db_id"]]\n            try:\n                if generator:\n                    answer, metadata = generator(row, path)\n                else:\n                    answer = saved[i]["prediction"]\n            except (ValueError, RuntimeError) as exc:\n                answer = ""\n                metadata["generation_error"] = str(exc)\n            record = {"index": i, "db_id": row["db_id"], "question": row["question"],\n                      "gold_sql": row["query"], "raw_prediction": answer, **metadata,\n                      **score_one(path, row["query"], answer, args.timeout, args.max_rows)}\n            results.append(record)\n            out.write(json.dumps(record) + "\\n")\n            out.flush()\n            # Single-line SQL for the official evaluator; sqlglot preserves quoted whitespace.\n            try:\n                single_line = parse_query(record["prediction"]).sql(dialect="sqlite", pretty=False)\n            except (ValueError, sqlglot.errors.SqlglotError):\n                single_line = "SELECT * FROM __invalid_prediction__"\n            sql_out.write(single_line.replace("\\n", " ") + "\\n")\n            gold_out.write(row["query"].replace("\\n", " ") + "\\t" + row["db_id"] + "\\n")\n            print(f"{len(results)}/{len(indices)} | match={record[\'execution_match\']}", flush=True)\n    n = len(results)\n    eligible = sum(r["gold_valid"] for r in results)\n    matches = sum(r["execution_match"] for r in results)\n    versions = {}\n    for package in ("sqlglot", "torch", "transformers", "peft", "datasets", "bitsandbytes"):\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    summary = {\n        "metric": "single_database_result_equivalence_not_official_spider_test_suite",\n        "samples": n, "gold_executable_samples": eligible, "gold_errors": n - eligible,\n        "syntax_validity": sum(r["syntax_valid"] for r in results) / n,\n        "execution_validity": sum(r["execution_valid"] for r in results) / n,\n        "execution_matches": matches, "execution_accuracy_all_samples": matches / n,\n        "execution_accuracy_gold_executable": matches / eligible if eligible else None,\n        "generation_errors": sum("generation_error" in r for r in results),\n        "evaluation_seconds_excluding_model_load": time.monotonic() - started,\n        "base_model": base, "versions": versions,\n        "arguments": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},\n        "comparison": "Column order and duplicates preserved; row order required only for outer gold ORDER BY; exact SQLite values, no float tolerance.",\n    }\n    if generator:\n        import torch\n        summary["peak_allocated_vram_gib"] = [torch.cuda.max_memory_allocated(i) / 2**30 for i in range(torch.cuda.device_count())]\n    (args.output / "summary.json").write_text(json.dumps(summary, indent=2) + "\\n")\n    print(json.dumps(summary, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')
(script_dir / 'benchmark_spider.py').write_text('#!/usr/bin/env python3\n"""Compare the unadapted base and final QLoRA adapter on identical Spider rows.\n\nDefault: 100 seeded validation samples, identical prompts, greedy generation,\nSQLite execution, and strict pandas.DataFrame.equals result comparison.\nRun with --help. See BENCHMARK.md for metric definitions and commands.\n"""\nimport argparse\nfrom contextlib import closing, nullcontext\nimport hashlib\nimport importlib.metadata\nimport json\nimport logging\nfrom pathlib import Path\nimport random\nimport sqlite3\nimport time\n\nimport pandas as pd\nimport sqlglot\n\nif __package__:\n    from .evaluate_spider import SYSTEM_PROMPT, database_path, extract_sql, parse_query, read_json_rows, schema_text\nelse:\n    from evaluate_spider import SYSTEM_PROMPT, database_path, extract_sql, parse_query, read_json_rows, schema_text\n\nLOG = logging.getLogger("spider-benchmark")\n\n\ndef execute_dataframe(path, sql, timeout=5.0, max_rows=100000):\n    """Execute one read-only query and preserve SQLite result labels/order."""\n    parse_query(sql)\n    start = time.monotonic()\n    allowed = {sqlite3.SQLITE_SELECT, sqlite3.SQLITE_READ,\n               sqlite3.SQLITE_FUNCTION, sqlite3.SQLITE_RECURSIVE}\n\n    def authorize(action, arg1, arg2, db, source):\n        if action == sqlite3.SQLITE_FUNCTION and (arg2 or "").lower() == "load_extension":\n            return sqlite3.SQLITE_DENY\n        return sqlite3.SQLITE_OK if action in allowed else sqlite3.SQLITE_DENY\n\n    with closing(sqlite3.connect(path.as_uri() + "?mode=ro", uri=True, timeout=timeout)) as conn:\n        conn.execute("PRAGMA query_only=ON")\n        conn.set_authorizer(authorize)\n        conn.set_progress_handler(lambda: int(time.monotonic() - start > timeout), 1000)\n        # Use a cursor to bound the result before constructing a DataFrame;\n        # pandas infers dtypes over the entire result, rather than per chunk.\n        cursor = conn.execute(sql)\n        rows = cursor.fetchmany(max_rows + 1)\n        if len(rows) > max_rows:\n            raise ValueError(f"Result exceeds row limit ({max_rows})")\n        return pd.DataFrame.from_records(rows, columns=[column[0] for column in cursor.description])\n\n\ndef score_prediction(path, answer, gold_df, timeout=5.0, max_rows=100000, generation_error=None):\n    result = {"sql": "", "syntax_valid": False, "execution_valid": False,\n              "execution_match": False, "error_type": None, "error": None}\n    if generation_error:\n        return {**result, "error_type": "generation", "error": generation_error}\n    try:\n        result["sql"] = extract_sql(answer)\n        parse_query(result["sql"])\n    except (ValueError, sqlglot.errors.SqlglotError) as exc:\n        return {**result, "error_type": "invalid_sql", "error": str(exc)}\n    result["syntax_valid"] = True\n    try:\n        predicted = execute_dataframe(path, result["sql"], timeout, max_rows)\n        result["execution_valid"] = True\n        result["execution_match"] = gold_df is not None and predicted.equals(gold_df)\n        result["result_shape"] = list(predicted.shape)\n    except sqlite3.Error as exc:\n        message = str(exc)\n        # SQLite uses SQLITE_ERROR for both grammar and name-resolution errors.\n        # Keep missing table/column errors separate from reported syntax errors.\n        is_syntax = any(s in message.lower() for s in (\n            "syntax error", "incomplete input", "unrecognized token"))\n        result.update(syntax_valid=not is_syntax,\n                      error_type="sqlite_syntax" if is_syntax else "sqlite_execution", error=message)\n    except (ValueError, sqlglot.errors.SqlglotError) as exc:\n        result.update(error_type="resource_or_query_limit", error=str(exc))\n    return result\n\n\nclass PairedGenerator:\n    """Keep one quantized base in memory; disable/enable LoRA for the comparison."""\n    def __init__(self, args):\n        import torch\n        from peft import PeftConfig, PeftModel\n        from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, set_seed\n\n        if not torch.cuda.is_available():\n            raise RuntimeError("CUDA is required for generation; use --predictions for CPU-only scoring.")\n        self.args, self.torch = args, torch\n        set_seed(args.seed)\n        config = PeftConfig.from_pretrained(args.adapter)\n        if config.peft_type != "LORA" or config.bias != "none" or config.modules_to_save:\n            raise ValueError("This paired comparison requires LoRA with bias=\'none\' and no modules_to_save.")\n        self.base_name = args.base_model or config.base_model_name_or_path\n        self.tokenizer = AutoTokenizer.from_pretrained(args.adapter)\n        LOG.info("Loading base %s in NF4; adapter=%s", self.base_name, args.adapter)\n        base = AutoModelForCausalLM.from_pretrained(\n            self.base_name, device_map={"": args.device}, torch_dtype=torch.float16,\n            quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",\n                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16))\n        self.model = PeftModel.from_pretrained(base, args.adapter, is_trainable=False).eval()\n        self.model_revision = getattr(self.model.config, "_commit_hash", None)\n\n    def __call__(self, row, path, use_adapter):\n        args, torch = self.args, self.torch\n        prompt = f"Database: {row[\'db_id\']}\\n"\n        if args.schema == "ddl":\n            prompt += "Schema:\\n" + schema_text(path) + "\\n"\n        prompt += "Question: " + row["question"]\n        messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]\n        inputs = self.tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True,\n                                                     return_tensors="pt", return_dict=True)\n        device = self.model.get_input_embeddings().weight.device\n        inputs = {k: v.to(device) for k, v in inputs.items()}\n        prompt_length = inputs["input_ids"].shape[1]\n        if prompt_length + args.max_new_tokens > args.max_context:\n            raise ValueError("Prompt plus generation budget exceeds --max-context; not truncated")\n        context = nullcontext() if use_adapter else self.model.disable_adapter()\n        start = time.monotonic()\n        with context, torch.no_grad():\n            tokens = self.model.generate(**inputs, do_sample=False, temperature=0.0, num_beams=1,\n                max_new_tokens=args.max_new_tokens, use_cache=True,\n                return_dict_in_generate=False, pad_token_id=self.tokenizer.eos_token_id)\n        generated = tokens[0, prompt_length:]\n        return self.tokenizer.decode(generated, skip_special_tokens=True), {\n            "seconds": time.monotonic() - start, "prompt_tokens": prompt_length,\n            "generated_tokens": len(generated), "hit_token_limit": len(generated) == args.max_new_tokens}\n\n\ndef aggregate(records):\n    n = len(records)\n    if not n:\n        raise ValueError("Cannot score an empty benchmark")\n    summary = {"samples": n, "gold_errors": sum(r["gold_error"] is not None for r in records),\n               "metric": "strict_pandas_dataframe_equals_single_database"}\n    for name in ("base", "finetuned"):\n        scores = [r[name] for r in records]\n        invalid = sum(not s["syntax_valid"] for s in scores)\n        summary[name] = {\n            "syntax_validity_pct": 100 * (n - invalid) / n,\n            "syntax_error_rate_pct": 100 * invalid / n,\n            "execution_validity_pct": 100 * sum(s["execution_valid"] for s in scores) / n,\n            "execution_accuracy_pct": 100 * sum(s["execution_match"] for s in scores) / n,\n            "syntax_invalid_count": invalid,\n            "generation_errors": sum(s["error_type"] == "generation" for s in scores),\n        }\n    base_errors = summary["base"]["syntax_invalid_count"]\n    ft_errors = summary["finetuned"]["syntax_invalid_count"]\n    summary["syntax_error_rate_reduction_pct"] = 100 * (base_errors - ft_errors) / base_errors if base_errors else None\n    summary["syntax_error_rate_drop_percentage_points"] = 100 * (base_errors - ft_errors) / n\n    summary["syntax_reduction_note"] = "Undefined when base error rate is zero; negative means regression."\n    return summary\n\n\ndef benchmark(rows, indices, paths, generator, output, timeout=5.0, max_rows=100000):\n    """Generator(row, path, use_adapter) returns (raw answer, metadata)."""\n    records = []\n    with (output / "predictions.jsonl").open("w") as stream:\n        for position, i in enumerate(indices, start=1):\n            row = rows[i]\n            path = paths[row["db_id"]]\n            record = {"index": i, "db_id": row["db_id"], "question": row["question"],\n                      "gold_sql": row["query"], "gold_error": None}\n            try:\n                gold_df = execute_dataframe(path, row["query"], timeout, max_rows)\n            except (sqlite3.Error, ValueError, sqlglot.errors.SqlglotError) as exc:\n                gold_df = None\n                record["gold_error"] = str(exc)\n                LOG.warning("Sample %s gold query failed: %s", i, exc)\n            for name in ("base", "finetuned"):\n                try:\n                    answer, metadata = generator({**row, "index": i}, path, name == "finetuned")\n                    error = metadata.get("generation_error")\n                except (RuntimeError, ValueError) as exc:\n                    answer, metadata, error = "", {}, str(exc)\n                score = score_prediction(path, answer, gold_df, timeout, max_rows, error)\n                record[name] = {**score, "raw_output": answer, "generation": metadata}\n                record[name + "_sql"] = score["sql"]\n                if score["error"]:\n                    LOG.warning("Sample %s %s %s: %s", i, name, score["error_type"], score["error"])\n            records.append(record)\n            stream.write(json.dumps(record) + "\\n")\n            stream.flush()\n            LOG.info("[%d/%d] db=%s | base EX=%s | fine-tuned EX=%s", position, len(indices),\n                     row["db_id"], record["base"]["execution_match"], record["finetuned"]["execution_match"])\n    return records\n\n\ndef save_results(records, output, metadata):\n    table = pd.DataFrame([{\n        "Index": r["index"], "Database": r["db_id"], "Question": r["question"],\n        "Gold SQL": r["gold_sql"], "Base Model SQL": r["base_sql"],\n        "Fine-Tuned Model SQL": r["finetuned_sql"],\n        "Base Execution Match": r["base"]["execution_match"],\n        "Fine-Tuned Execution Match": r["finetuned"]["execution_match"],\n        "Base Syntax Valid": r["base"]["syntax_valid"],\n        "Fine-Tuned Syntax Valid": r["finetuned"]["syntax_valid"],\n        "Base Error": r["base"]["error"], "Fine-Tuned Error": r["finetuned"]["error"],\n        "Gold Error": r["gold_error"],\n    } for r in records])\n    table.to_csv(output / "comparison.csv", index=False)\n    summary = {**aggregate(records), "metadata": metadata}\n    (output / "summary.json").write_text(json.dumps(summary, indent=2) + "\\n")\n    (output / "comparison.html").write_text(\n        \'<!doctype html><html><head><meta charset="utf-8"><title>Spider benchmark</title>\'\n        \'<style>body{font-family:system-ui}table{border-collapse:collapse}td,th{padding:8px;border:1px solid #ccc;\'\n        \'vertical-align:top}td{min-width:120px;max-width:380px;overflow-wrap:anywhere}</style></head><body>\'\n        \'<h1>Base versus fine-tuned Spider benchmark</h1><p>Strict Pandas DataFrame equality; \'\n        \'single-database evaluation.</p>\' + table.to_html(index=False, escape=True) + \'</body></html>\')\n    LOG.info("Base EX: %.2f%% | Fine-tuned EX: %.2f%% | Syntax error reduction: %s",\n             summary["base"]["execution_accuracy_pct"], summary["finetuned"]["execution_accuracy_pct"],\n             "N/A (base error rate zero)" if summary["syntax_error_rate_reduction_pct"] is None\n             else f"{summary[\'syntax_error_rate_reduction_pct\']:.2f}%")\n    print(table.head(5).to_string(index=False, max_colwidth=70))\n    return summary\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--adapter", default="results_3epochs/lora_sql_adapter_3epochs")\n    parser.add_argument("--base-model", help="Defaults to adapter\'s saved base model name")\n    parser.add_argument("--db-root", type=Path, required=True)\n    parser.add_argument("--data", type=Path, help="Local Spider dev.json; otherwise HF validation split")\n    parser.add_argument("--dataset-revision", default="main")\n    parser.add_argument("--samples", type=int, default=100)\n    parser.add_argument("--seed", type=int, default=3407)\n    parser.add_argument("--schema", choices=["none", "ddl"], default="none")\n    parser.add_argument("--device", default="cuda:0")\n    parser.add_argument("--max-new-tokens", type=int, default=256)\n    parser.add_argument("--max-context", type=int, default=2048)\n    parser.add_argument("--timeout", type=float, default=5.0)\n    parser.add_argument("--max-rows", type=int, default=100000)\n    parser.add_argument("--predictions", type=Path, help="CPU rescore JSON/JSONL: index, db_id, question, base_sql, finetuned_sql")\n    parser.add_argument("--output", type=Path, default=Path("spider_benchmark"))\n    args = parser.parse_args()\n    for key in ("samples", "max_new_tokens", "max_context", "timeout", "max_rows"):\n        if getattr(args, key) <= 0:\n            parser.error(f"--{key.replace(\'_\', \'-\')} must be positive")\n    if args.output.exists() and any(args.output.iterdir()):\n        parser.error("Output directory must be empty; choose a new --output to preserve prior runs")\n    if args.data:\n        rows = read_json_rows(args.data)\n    else:\n        from datasets import load_dataset\n        rows = list(load_dataset("xlangai/spider", split="validation", revision=args.dataset_revision))\n    if args.samples > len(rows):\n        parser.error("--samples exceeds the validation split size")\n    indices = sorted(random.Random(args.seed).sample(range(len(rows)), args.samples))\n    paths = {rows[i]["db_id"]: database_path(args.db_root, rows[i]["db_id"]) for i in indices}\n    args.output.mkdir(parents=True, exist_ok=True)\n    logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s",\n                        handlers=[logging.StreamHandler(), logging.FileHandler(args.output / "benchmark.log")])\n    LOG.info("Selected %d validation samples, seed=%d, schema=%s", len(indices), args.seed, args.schema)\n    if args.predictions:\n        saved = {}\n        for item in read_json_rows(args.predictions):\n            if item["index"] in saved:\n                parser.error(f"Duplicate prediction index {item[\'index\']}")\n            saved[item["index"]] = item\n        for i in indices:\n            if i not in saved or any(saved[i].get(k) != rows[i][k] for k in ("db_id", "question")):\n                parser.error(f"Missing or misaligned predictions at index {i}")\n            if any(not isinstance(saved[i].get(k), str) for k in ("base_sql", "finetuned_sql")):\n                parser.error(f"SQL predictions must be strings at index {i}")\n\n        def generator(row, path, use_adapter):\n            name = "finetuned" if use_adapter else "base"\n            item = saved[row["index"]]\n            prior = item.get(name, {})\n            # Keep raw answers and generation failures when rescoring our JSONL.\n            metadata = {}\n            if prior.get("error_type") == "generation":\n                metadata["generation_error"] = prior["error"]\n            return prior.get("raw_output", item[name + "_sql"]), metadata\n    else:\n        generator = PairedGenerator(args)\n    started = time.monotonic()\n    records = benchmark(rows, indices, paths, generator, args.output, args.timeout, args.max_rows)\n    versions = {}\n    for package in ("pandas", "sqlglot", "torch", "transformers", "peft", "bitsandbytes", "datasets"):\n        try:\n            versions[package] = importlib.metadata.version(package)\n        except importlib.metadata.PackageNotFoundError:\n            pass\n    metadata = {\n        "arguments": {k: str(v) if isinstance(v, Path) else v for k, v in vars(args).items()},\n        "sample_indices": indices, "base_model": getattr(generator, "base_name", None),\n        "base_revision": getattr(generator, "model_revision", None), "versions": versions,\n        "sample_sha256": hashlib.sha256(json.dumps([rows[i] for i in indices], sort_keys=True).encode()).hexdigest(),\n        "seconds_excluding_model_load": time.monotonic() - started,\n        "generation": {"do_sample": False, "temperature": 0.0, "num_beams": 1},\n        "comparison": "DataFrame.equals: exact labels, row/column order, dtypes and values; no sorting or alias normalization.",\n    }\n    save_results(records, args.output, metadata)\n\n\nif __name__ == "__main__":\n    main()\n')
sys.path.insert(0, str(script_dir))


In [ ]:
import hashlib, json, random, time, zipfile
import torch
from datasets import load_dataset
from huggingface_hub import HfApi
from benchmark_spider import execute_dataframe

assert torch.cuda.is_available(), "Kaggle CUDA GPU not attached"
print("GPU:", torch.cuda.get_device_name(0), flush=True)
input_root = Path("/kaggle/input")
adapters = list(input_root.rglob("lora_sql_adapter_3epochs/adapter_model.safetensors"))
assert len(adapters) == 1, f"Expected one final adapter, found {adapters}"
adapter = adapters[0].parent
with adapters[0].open("rb") as stream:
    adapter_sha = hashlib.file_digest(stream, "sha256").hexdigest()
assert adapter_sha == "961d13a4c49a15b92e1a0801f72e86253f4c4f5f7665786e9e758fd6ad1f5a9a", "Unexpected adapter weights"
db_candidates = list(input_root.rglob("concert_singer/concert_singer.sqlite"))
assert len(db_candidates) == 1, f"Ambiguous database root: {db_candidates}"
db_root = db_candidates[0].parent.parent
revision = HfApi().dataset_info("xlangai/spider").sha
rows = list(load_dataset("xlangai/spider", split="validation", revision=revision))
indices = sorted(random.Random(3407).sample(range(len(rows)), 100))
data_path = Path("/kaggle/working/validation.json")
data_path.write_text(json.dumps(rows) + "\n")
db_hashes = {}
for i in indices:
    row = rows[i]
    db = db_root / row["db_id"] / (row["db_id"] + ".sqlite")
    execute_dataframe(db, row["query"])
    if row["db_id"] not in db_hashes:
        with db.open("rb") as stream:
            db_hashes[row["db_id"]] = hashlib.file_digest(stream, "sha256").hexdigest()
print(f"Preflight passed: {len(indices)} gold queries, {len(db_hashes)} databases", flush=True)
with zipfile.ZipFile("/kaggle/working/benchmark_databases.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for db_id in db_hashes:
        db = db_root / db_id / (db_id + ".sqlite")
        archive.write(db, f"database/{db_id}/{db_id}.sqlite")
provenance = {"adapter_sha256": adapter_sha, "adapter_source": "yettouabdallah/qlora-text2sql-3epochs",
    "database_source": "yousufqureshi5/spider-sqlite-databases", "database_sha256": db_hashes,
    "dataset": "xlangai/spider", "dataset_revision": revision, "samples": 100, "seed": 3407,
    "schema_mode": "none", "gpu": torch.cuda.get_device_name(0),
    "torch_version": torch.__version__, "sample_indices": indices,
    "script_sha256": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in script_dir.glob("*.py")}}
Path("/kaggle/working/run_provenance.json").write_text(json.dumps(provenance, indent=2))
started = time.monotonic()
subprocess.run([sys.executable, str(script_dir / "benchmark_spider.py"),
    "--adapter", str(adapter), "--db-root", str(db_root), "--data", str(data_path),
    "--samples", "100", "--seed", "3407", "--schema", "none", "--device", "cuda:0",
    "--output", "/kaggle/working/spider_benchmark_100"], check=True)
summary = json.loads(Path("/kaggle/working/spider_benchmark_100/summary.json").read_text())
summary["kaggle_run_seconds_including_model_load"] = time.monotonic() - started
Path("/kaggle/working/spider_benchmark_100/summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2), flush=True)
assert summary["gold_errors"] == 0
assert summary["base"]["generation_errors"] == 0 and summary["finetuned"]["generation_errors"] == 0, "Generation errors require investigation"
